# 2단계 — attention 관측 (NIAR)

1단계(실험 1)에서 확인한 행동 — context 내 위반 코드가 후속 함수의 준수율을 낮춘다 —
이 일어날 때, 모델이 실제로 **위반 코드 구간을 더 참조(attention 배분)** 하는지 관측한다.

관련 계획서 절: **3.4**(attention 꺼내기) · **3.7**(NIAR 지표).

> 이건 **상관 관측**이지 인과 주장이 아니다. 2단계 가설 **H5/H6은 탐색적 보고**
> (주 가설은 H1a·H2a·H3 뿐). 인과 주장은 전적으로 3단계 개입에 근거한다.

진행 순서: **① 512토큰 eager 대조 검증(게이트) → ② NIAR 관측**.

## 0. GPU 확인
양자화 없이 fp16으로 돌린다(개입 실험과의 연속성). 3B fp16은 T4(16GB)에 올라간다.
**GPU가 안 뜨면** 런타임 → 런타임 유형 변경 → T4 GPU 로 바꿔라(관측은 GPU 필수).

In [ ]:
!nvidia-smi -L

## 1. repo clone (main)
코드는 **main**에서 받는다. 이미 clone돼 있으면 main 최신으로 맞춘다
(옛 브랜치에 고정돼 낡은 코드가 도는 사고 방지).

In [ ]:
REPO_URL = "https://github.com/deanjs/instruction-adherence.git"
BRANCH = "main"
import os
# 재시작 후에도 결과 JSONL을 보존하려고, 이미 clone돼 있으면 지우지 않는다.
if not os.path.isdir("instruction-adherence"):
    !git clone --branch {BRANCH} {REPO_URL}
%cd instruction-adherence
# 이미 있던 clone도 main 최신으로 강제 정렬(코드만; 결과는 --out으로 Drive에 둔다)
!git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -1

## 2. 의존성
Colab에는 torch가 사전 설치돼 있다(그 CUDA 빌드를 유지해야 하므로 재설치하지 않는다).
transformers/accelerate만 버전을 맞춘다. `AttentionInterface` 등록에 transformers>=4.51 필요.

In [ ]:
!pip install -q "transformers>=4.51.0" "accelerate>=0.26.0"
import torch, transformers
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

## 3. 검증 (게이트) — 512토큰 eager 대조

prefill과 decode가 **다른 커널**을 쓰므로, 캡처 축약값이 `output_attentions=True` eager
결과와 수치가 일치하는지 확인한다(계획서 3.4). **PASS여야 관측으로 넘어간다.**

검증은 커널 등가성만 보므로 작은 모델(1.5B)·fp32로 tight tolerance.

In [ ]:
!python src/stage2_attention.py --validate

## 4. 관측 — NIAR / NSCAR / NVCAR

exp1 확증과 **동일한 조건(compliant_remaining ∈ {4,3,2,1,0})·동일 seed**로,
지침/선행코드/위반코드 구간의 길이 정규화 attention 배분을 잰다.
결과는 Drive의 `stage2_niar.jsonl`에 세션 단위로 append(재개 가능).

- 관측은 기본 **greedy**(재현성). exp1과 동일 샘플링을 보려면 `--do-sample`.
- 기본 모델은 exp1과 같은 3B. `--model`로 교체 가능.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/instruction-adherence/stage2_niar.jsonl"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!python src/stage2_attention.py --observe --n-seeds 20 --out "{OUT}"

## 5. 집계
조건별 평균 NIAR(지침)·NSCAR(선행코드)·NVCAR(위반코드). >1이면 평균보다 많이 참조.

In [ ]:
!python src/stage2_attention.py --observe --summary-only --out "{OUT}"

## 5-b. 층별 진단
전체 평균이 조건 간 평탄해도 **특정 층에서 갈리는지** 본다.
저장된 `layer_profile`만 읽으므로 재실행 불필요. |Δ| 큰 층이 있으면 그 층에 조건 신호.

In [ ]:
!python src/stage2_attention.py --observe --layer-summary --out "{OUT}"

## 6. 결과 내려받기 (선택)
`stage2_niar.jsonl`은 사전 등록 기록이다. Drive에 이미 있으니 이 셀은 로컬 커밋용.

In [ ]:
from google.colab import files
files.download(OUT)